## Retrieval-Augmented Generation (RAG)

Large Language Models (LLMs) generate responses primarily from knowledge acquired during training. Consequently, they do not automatically have access to specialised or private collections, such as a locally stored research corpus. **Retrieval-Augmented Generation (RAG)** addresses this limitation by combining an LLM with an external information retrieval system.[^1]

Instead of relying exclusively on the model's internal knowledge, a RAG system first **retrieves information relevant to the user's query** and then provides this information to the LLM as additional context. This is particularly useful when working with specialised corpora that were not part of the model's training data, or collections that are too large to fit into the model's context window.[^1]

### A simplified RAG pipeline can be represented as:

> **User question → retrieve relevant documents → add documents to the context → LLM → generated answer**

RAG does not normally retrain the language model on the external collection. Instead, the external data are made available to the model **at inference time**.[^1]




In [ ]:
%pip install --upgrade --force-reinstall \
    "pydantic>=2.12,<2.13" \
    langchain \
    langchain-openai \
    langchain-chroma \
    langchain-docling \
    langchain-community \
    langchain-text-splitters \
    chromadb \
    docling \
    beautifulsoup4 \
    "numpy<2" \
    "pandas>=2.2,<3"

  Using cached langchain-1.4.0-py3-none-any.whl.metadata (6.2 kB)
  Using cached langchain_openai-1.6.2-py3-none-any.whl.metadata (3.4 kB)
  Using cached langchain_chroma-1.1.0-py3-none-any.whl.metadata (1.9 kB)
  Using cached langchain_docling-3.0.0-py3-none-any.whl.metadata (5.8 kB)
  Using cached langchain_community-0.4.2-py3-none-any.whl.metadata (3.4 kB)
  Using cached langchain_text_splitters-1.1.2-py3-none-any.whl.metadata (3.3 kB)
  Using cached chromadb-1.5.9-cp39-abi3-macosx_11_0_arm64.whl.metadata (5.0 kB)
  Using cached docling-2.127.0-py3-none-any.whl.metadata (11 kB)
  Using cached beautifulsoup4-4.15.0-py3-none-any.whl.metadata (3.8 kB)
  Using cached numpy-1.26.4-cp312-cp312-macosx_11_0_arm64.whl.metadata (61 kB)
  Using cached pandas-2.3.3-cp312-cp312-macosx_11_0_arm64.whl.metadata (91 kB)
  Using cached typing_extensions-4.16.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached typing_inspection-0.4.4-py3-none-any.whl.metadata (2.6 kB)
  Using cached langchain_core-1.6

### Installations (skip if not neccessary)

In [3]:
import sys
!{sys.executable} -m pip install -U langchain-community

In [1]:
import importlib.metadata as md

for package in [
    "docling",
    "langchain-docling",
    "pydantic",
    "langchain",
    "numpy",
]:
    print(package, md.version(package))

from langchain_docling import DoclingLoader

print("Docling import successful")

docling 2.127.0
langchain-docling 3.0.0
pydantic 2.8.2
langchain 1.4.0
numpy 1.26.4


/opt/anaconda3/lib/python3.12/site-packages/pydantic/_internal/_fields.py:161: UserWarning: Field "model_impl" has conflict with protected namespace "model_".

You may be able to resolve this warning by setting `model_config['protected_namespaces'] = ()`.
  warnings.warn(
/opt/anaconda3/lib/python3.12/site-packages/pydantic/_internal/_fields.py:161: UserWarning: Field "model_spec" has conflict with protected namespace "model_".

You may be able to resolve this warning by setting `model_config['protected_namespaces'] = ()`.
  warnings.warn(
/opt/anaconda3/lib/python3.12/site-packages/pydantic/_internal/_fields.py:161: UserWarning: Field "model_name" has conflict with protected namespace "model_".

You may be able to resolve this warning by setting `model_config['protected_namespaces'] = ()`.
  warnings.warn(
/opt/anaconda3/lib/python3.12/site-packages/pydantic/_internal/_fields.py:161: UserWarning: Field "model_version" has conflict with protected namespace "model_".

You may be able to

AttributeError: force_full_page_ocr

In [2]:
import sys
import importlib.metadata as md

print("Python:", sys.executable)

for package in ["docling", "langchain-docling", "pydantic", "pydantic-core"]:
    try:
        print(f"{package}: {md.version(package)}")
    except md.PackageNotFoundError:
        print(f"{package}: NOT INSTALLED")

Python: /opt/anaconda3/bin/python
docling: 2.127.0
langchain-docling: 3.0.0
pydantic: 2.8.2
pydantic-core: 2.20.1


In [4]:
import sys
import numpy as np

print(sys.executable)
print(np.__version__)
print(np.__file__)

/opt/anaconda3/bin/python
1.26.4
/opt/anaconda3/lib/python3.12/site-packages/numpy/__init__.py


In [3]:
import sys

!{sys.executable} -m pip install \
    --upgrade \
    --force-reinstall \
    --no-cache-dir \
    "pydantic==2.13.5"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 10.9 MB/s eta 0:00:00
  Attempting uninstall: typing-extensions
    Found existing installation: typing_extensions 4.16.0
    Uninstalling typing_extensions-4.16.0:
      Successfully uninstalled typing_extensions-4.16.0
  Attempting uninstall: annotated-types
    Found existing installation: annotated-types 0.6.0
    Uninstalling annotated-types-0.6.0:
      Successfully uninstalled annotated-types-0.6.0
  Attempting uninstall: typing-inspection
    Found existing installation: typing-inspection 0.4.4
    Uninstalling typing-inspection-0.4.4:
      Successfully uninstalled typing-inspection-0.4.4
  Attempting uninstall: pydantic-core
    Found existing installation: pydantic_core 2.20.1
    Uninstalling pydantic_core-2.20.1:
      Successfully uninstalled pydantic_core-2.20.1
  Attempting uninstall: pydantic
    Found existing installation: pydantic 2.8.2
    Uninstalling pydantic-2.8.2:
      Successfully uninstalled pydantic-2.8

In [1]:
import sys
import pydantic
import importlib.metadata as md

print("Python:", sys.executable)
print("Pydantic:", pydantic.__version__)
print("Pydantic location:", pydantic.__file__)
print("Docling:", md.version("docling"))
print("LangChain Docling:", md.version("langchain-docling"))

from langchain_docling import DoclingLoader

print("Docling import successful")

Python: /opt/anaconda3/bin/python
Pydantic: 2.13.5
Pydantic location: /opt/anaconda3/lib/python3.12/site-packages/pydantic/__init__.py
Docling: 2.127.0
LangChain Docling: 3.0.0
Docling import successful


In [3]:
import sys
!{sys.executable} -m pip check

mdit-py-plugins 0.3.0 has requirement markdown-it-py<3.0.0,>=1.0.0, but you have markdown-it-py 4.2.0.
torchaudio 2.9.0 has requirement torch==2.9.0, but you have torch 2.14.0.
gensim 4.3.3 has requirement scipy<1.14.0,>=1.7.0, but you have scipy 1.17.1.
s3fs 2024.6.1 has requirement fsspec==2024.6.1.*, but you have fsspec 2026.7.0.
thinc 8.3.6 has requirement numpy<3.0.0,>=2.0.0, but you have numpy 1.26.4.
streamlit 1.37.1 has requirement packaging<25,>=20, but you have packaging 26.3.
streamlit 1.37.1 has requirement pillow<11,>=7.1.0, but you have pillow 12.3.0.
streamlit 1.37.1 has requirement protobuf<6,>=3.20, but you have protobuf 7.36.1.
streamlit 1.37.1 has requirement rich<14,>=10.14.0, but you have rich 15.0.0.
streamlit 1.37.1 has requirement tenacity<9,>=8.1.0, but you have tenacity 9.1.4.


### Main imports

In [4]:
import os
import warnings
import logging

import bs4

from langchain.agents import AgentState, create_agent
from langchain.messages import MessageLikeRepresentation
from langchain.tools import tool

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_docling import DoclingLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

### Environment setup

For this step you will need to: 
- get a Langchain API Key (https://docs.langchain.com/oss/python/deepagents/rag)
- be added DHInfra project by Florian and get DHInfa API kez

In [5]:
from openai import OpenAI

In [7]:
os.environ["LANGCHAIN_API_KEY"] = "" # insert your own Langchain key
os.environ["LANGCHAIN_TRACING_V2"] = "false"  # <-- FIX 1: Disabled to prevent 403 error
os.environ["LANGCHAIN_PROJECT"] = "DHInfra-Tracing-Demo"
os.environ["LANGSMITH_DISABLE_RUN_COMPRESSION"] = "true"
os.environ["USER_AGENT"] = "my_agent"
os.environ["DHINFRA_API_KEY"] = "" # insert DHInfra key

In [8]:
# <-- FIX 2: Custom class to prevent the 422 "null content" error
class SanitizedChatOpenAI(ChatOpenAI):
    def _get_request_payload(self, input_, *args, **kwargs):
        payload = super()._get_request_payload(input_, *args, **kwargs)
        if "messages" in payload:
            for msg in payload["messages"]:
                if msg.get("content") is None:
                    msg["content"] = ""
        return payload

# Initialize chat model using the sanitized class
model = SanitizedChatOpenAI(
    model="qwen3.5-397b",
    openai_api_key=os.environ["DHINFRA_API_KEY"],
    openai_api_base="https://api.dhinfra.uni-graz.at/v1",
    model_kwargs={"parallel_tool_calls": False}
)

# Initialize embedding model
embeddings = OpenAIEmbeddings(
    model="qwen3-embedding-8b",
    openai_api_key=os.environ["DHINFRA_API_KEY"],
    openai_api_base="https://api.dhinfra.uni-graz.at/v1"
)

vector_store = Chroma(
    collection_name="migraanno_newspapers_v2",
    embedding_function=embeddings,
    persist_directory="./chroma_migraanno",
)


print("Chat model (Qwen), embedding model (Qwen3-Embedding-8B), and Chroma vector store setup done")

Chat model (Qwen), embedding model (Qwen3-Embedding-8B), and Chroma vector store setup done


In [9]:
print("Documents in Chroma:", vector_store._collection.count())

Documents in Chroma: 96871


## Why metadata filtering works

The following filter works because metadata was attached to each newspaper document **before the documents were indexed in Chroma**:

```python
records_1889 = vector_store.get(
    where={"year": 1889},
    include=["metadatas"],
)
```

During preprocessing (previous notebook), every CSV row was converted into a LangChain `Document`. Each document contained two components:

* `page_content`: the text used to create the embedding and perform semantic retrieval;
* `metadata`: structured fields used for exact filtering and for describing retrieved results.

```python
def clean_value(value):
    return "" if pd.isna(value) else value


docs = []

for _, row in df.iterrows():

    # Text used for semantic retrieval and embedding.
    page_content = f"""
Topic: {clean_value(row["Name-original"])}
Sentiment: {clean_value(row["sentiment"])}
Category: {clean_value(row["Category"])}
Year: {clean_value(row["year"])}

{clean_value(row["text"])}
""".strip()

    # Structured metadata used for filtering and result interpretation.
    metadata = {
        "id": clean_value(row["id"]),
        "topic": clean_value(row["Topic"]),
        "name_original": clean_value(row["Name-original"]),
        "newspaper_title": clean_value(row["newspaper_title"]),
        "date": clean_value(row["date"]),
        "preceding_document": clean_value(row["preceding_document"]),
        "following_document": clean_value(row["following_document"]),
        "relevancy_proba": clean_value(row["Relevancy_proba"]),
        "sentiment": clean_value(row["sentiment"]),
        "year": clean_value(row["year"]),
        "category": clean_value(row["Category"]),
    }

    # Attach the text and metadata to the same Document object.
    docs.append(
        Document(
            page_content=page_content,
            metadata=metadata,
        )
    )
```

The metadata was therefore stored inside every item in `docs`:

```python
docs[0].page_content
docs[0].metadata
```

Before indexing, `filter_complex_metadata()` removed metadata values that Chroma could not store. Supported scalar values, such as strings, integers, and floats, were retained.

```python
import random

print("Filtering metadata...")

filtered_docs = filter_complex_metadata(docs)

# Select up to 10,000 documents for indexing.
filtered_docs = random.sample(
    filtered_docs,
    min(10_000, len(filtered_docs)),
)

print(f"Filtering done: {len(filtered_docs)} documents")
```

Finally, `add_documents()` stored each document’s text, embedding, and metadata in the persistent Chroma collection:

```python
print("Starting vector store indexing...")

batch_size = 100

for start in range(0, len(filtered_docs), batch_size):
    end = min(start + batch_size, len(filtered_docs))

    vector_store.add_documents(
        documents=filtered_docs[start:end]
    )

    print(f"Indexed {end}/{len(filtered_docs)} documents")
```

The relevant data flow is:

> CSV row → `Document(page_content, metadata)` → metadata filtering → Chroma indexing

Consequently, Chroma can later filter the saved collection by fields such as:

```python
{"year": 1889}
{"category": "MIN"}
{"sentiment": "negative"}
```

`filter_complex_metadata()` does not select documents by year, category, or sentiment. It only ensures that their metadata values are compatible with Chroma. The actual metadata is persisted when the `Document` objects are passed to `vector_store.add_documents()`.


In [13]:
records_1889 = vector_store.get(
    where={"year": 1889},
    include=["metadatas"],
)

print("Documents from 1889:", len(records_1889["ids"]))
print(records_1889["metadatas"][:3])

Documents from 1889: 748
[{'relevancy_proba': 0.9999347, 'sentiment': 'neutral', 'year': 1889, 'preceding_document': 'Und so wird es bleiben, so lange die Bäcker und so viele andere Arbeiter zu Hause, im eigenen Lande, sich Zustände gefallen lassen, gegen welche fortgeschrittene Arbeiter nicht nur in Worten, sondern mit Thaten protestiren würden. Ueberflüssige Kopfschmerzen macht unseren offiziösen Blättern das Zwanzigjährige Gründungsfest des Arbeiter=Bildungsvereins „Vorwärts“ in Preßburg.', 'following_document': 'Selbst diesem harmlosen Blättchen war die Geschichte zu toll, wie die folgenden Notizen zeigen, die zwischen dem 12. und 15. September nacheinander erschienen. Näherer! Die offiziösen Wiener Journale „Presse“ und „Fremdenblatt“ enthalten in ihren gestrigen Frühnummern ein Telegramm aus Preßburg folgenden Inhaltes: „Die in Wien verbotene Lassalle=Feier soll durch Wiener Arbeiter hier im nächsten Monat begangen werden.', 'chunk_id': '189.0', 'newspaper_title': 'aze', 'date': 

## Search newspapers function

## Newspaper Search Function

The `search_newspapers()` function performs **semantic retrieval** over the newspaper texts stored in Chroma while optionally applying exact metadata filters.

```python
def search_newspapers(
    query: str,
    year: int | None = None,
    sentiment: str | None = None,
    category: str | None = None,
    k: int = 10,
):
    ...
```

### Parameters

| Parameter | Type | Default | Description |
|---|---:|---:|---|
| `query` | `str` | Required | Semantic search query. It may contain English, German, or multilingual search terms. |
| `year` | `int \| None` | `None` | Restricts retrieval to documents from one exact year. |
| `sentiment` | `str \| None` | `None` | Restricts retrieval to an exact sentiment label, such as `"negative"`, `"neutral"`, or `"positive"`. The value is converted to lowercase. |
| `category` | `str \| None` | `None` | Restricts retrieval to an exact category value, such as `"MIG"` or `"CTX"`. |
| `k` | `int` | `10` | Maximum number of documents returned. |

### Retrieval procedure

The function combines two retrieval mechanisms:

1. **Semantic similarity search**  
   Chroma compares the meaning of `query` with the embedded newspaper texts.

2. **Metadata filtering**  
   The optional `year`, `sentiment`, and `category` parameters restrict the documents considered during semantic search.

When several filters are supplied, they are combined with Chroma's `$and` operator. Therefore, a document must satisfy all specified conditions.

### Return value

The function returns a list of LangChain `Document` objects:

```python
list[Document]
```

Each result contains:

- `result.page_content`: the retrieved newspaper text;
- `result.metadata`: associated metadata such as date, year, newspaper, category, sentiment, and neighbouring text.

### Examples

#### Semantic search without metadata filters

```python
results = search_newspapers(
    query="minority populations and ethnic groups",
    k=10,
)
```

#### Search within one year

```python
results = search_newspapers(
    query="minority populations and ethnic groups",
    year=1889,
    k=10,
)
```

#### Search by year and sentiment

```python
results = search_newspapers(
    query="hostility towards ethnic minorities",
    year=1889,
    sentiment="negative",
    k=20,
)
```

#### Search by year, sentiment, and category

```python
results = search_newspapers(
    query="Croats and Serbs, Kroaten und Serben, Nationalitätenfrage",
    year=1889,
    sentiment="negative",
    category="MIG",
    k=20,
)
```

### Important notes

- Metadata filters use **exact matching**.
- `"MIG"` and `"mig"` are different category values unless categories were normalized before indexing.
- The year must be stored in Chroma as an integer for `year=1889` to match.
- `k` specifies the maximum number of returned results, not necessarily the total number of matching documents.
- English queries can retrieve German texts when a multilingual embedding model is used.
- Including German terms and historical spelling variants can improve retrieval from historical newspapers.

In [15]:
from typing import Any
from langchain_core.documents import Document


def search_newspapers(
    query: str,
    # query must be one string, not a tuple or list.

    year: int | None = None,
    # Either an integer such as 1889 or None when no year filter is wanted.

    sentiment: str | None = None,
    # Either a string such as "negative" or None.

    category: str | None = None,
    # Either a string such as "MIN" or None.

    k: int = 10,
    # Integer specifying the maximum number of returned documents.

) -> list[Document]:
    """
    Perform semantic retrieval with optional exact metadata filtering.

    Returns:
        A list of LangChain Document objects. Each Document contains
        page_content (str) and metadata (dict).
    """

    # A list containing Chroma filter dictionaries.
    # Initially empty because all metadata filters are optional.
    conditions: list[dict[str, Any]] = []

    # `is not None` distinguishes a missing year from a supplied year.
    if year is not None:

        # Convert values such as "1889" or 1889.0 to the integer 1889.
        normalized_year: int = int(year)

        # Create an exact-equality Chroma condition:
        # stored metadata["year"] must equal normalized_year.
        conditions.append({
            "year": {"$eq": normalized_year}
        })

    # Empty strings and None both evaluate to False.
    if sentiment:

        # Remove surrounding whitespace and normalize capitalization.
        # Example: " Negative " becomes "negative".
        normalized_sentiment: str = sentiment.strip().lower()

        # This remains an exact string comparison.
        conditions.append({
            "sentiment": {"$eq": normalized_sentiment}
        })

    if category:

        # Remove surrounding whitespace.
        # Capitalization is preserved:
        # "MIN" and "min" remain different values.
        normalized_category: str = category.strip()

        # Require an exact match with metadata["category"].
        conditions.append({
            "category": {"$eq": normalized_category}
        })

    # Chroma accepts either a filter dictionary or None.
    metadata_filter: dict[str, Any] | None

    if len(conditions) == 0:
        # No metadata filters: search all stored documents.
        metadata_filter = None

    elif len(conditions) == 1:
        # Chroma accepts one condition directly.
        # Example: {"year": {"$eq": 1889}}
        metadata_filter = conditions[0]

    else:
        # Combine multiple conditions using logical AND.
        # Every condition must be true for a document to be considered.
        metadata_filter = {
            "$and": conditions
        }

    # Embed `query` as a vector using the embedding model connected
    # to vector_store.
    #
    # Chroma first restricts candidate documents using metadata_filter,
    # then ranks those candidates by vector similarity to the query.
    results: list[Document] = vector_store.similarity_search(
        query=query,              # str: text converted into a query embedding
        k=int(k),                 # int: maximum number of returned documents
        filter=metadata_filter,   # dict or None: exact metadata restrictions
    )

    # Each returned item has:
    # result.page_content -> str
    # result.metadata     -> dict
    return results

## Query 1 

In [16]:
results = vector_store.similarity_search(
    query=(
        "minorities, ethnic minorities, linguistic minorities, "
        "religious minorities, minority populations"
    ),
    k=10,
    filter={"year": 1889},
)

In [17]:
for number, result in enumerate(results, start=1):
    print("=" * 80)
    print(f"Result {number}")
    print("ID:", result.metadata.get("chunk_id"))
    print("Date:", result.metadata.get("date"))
    print("Newspaper:", result.metadata.get("newspaper_title"))
    print("Topic:", result.metadata.get("topic"))
    print("Category:", result.metadata.get("category"))
    print("Sentiment:", result.metadata.get("sentiment"))
    print()
    print(result.page_content[:800])

Result 1
ID: 220165.0
Date: 1889-11-01
Newspaper: nfp
Topic: 7
Category: CTX
Sentiment: positive

Topic: 7
Original label: 7_wahlen_wähler_abstimmung_wahl
Category: CTX
Sentiment: positive

Diesen Trost gewährt die diesmalige Budget=Debatte des Reichstages, und dadurch empfängt sie ihre Bedeutung. Es ist von allen Parteien wie vom Regierungstische hinausgesprochen worden zu dem Volke; gute Vorsätze und verschämte Bekenntnisse, scharfe Anschuldigungen und beschwichtigende Friedensbethenerungen, kühne Beweisführungen und mühsame Widerlegungen — die bunte Fülle wird sich in dem Intellecte des Wählers von selbst sichten, und erst am Wahltage wird um das Echo der Reden vernehmen, welche seit vorgestern auf den deutschen Reichstag die allgemeine Aufmerksamkeit gelenkt haben.)
Result 2
ID: 118331.0
Date: 1889-12-01
Newspaper: vtl
Topic: 13
Category: MIN
Sentiment: negative

Topic: 13
Original label: 13_juden_deutschen_jüdischen_jüdische
Category: MIN
Sentiment: negative

Das Factum, daß ein K

## Query 2

In [18]:
for number, result in enumerate(results, start=1):
    metadata = result.metadata

    print("=" * 100)
    print(f"Result {number}")
    print("ID:", metadata.get("chunk_id"))
    print("Date:", metadata.get("date"))
    print("Year:", metadata.get("year"))
    print("Newspaper:", metadata.get("newspaper_title"))
    print("Topic:", metadata.get("topic"))
    print("Category:", metadata.get("category"))
    print("Sentiment:", metadata.get("sentiment"))

    print("\n--- PRECEDING TEXT ---")
    print(metadata.get("preceding_document") or "[No preceding text]")

    print("\n--- RETRIEVED TEXT ---")
    print(result.page_content or "[No retrieved text]")

    print("\n--- FOLLOWING TEXT ---")
    print(metadata.get("following_document") or "[No following text]")

    print()

Result 1
ID: 220165.0
Date: 1889-11-01
Year: 1889
Newspaper: nfp
Topic: 7
Category: CTX
Sentiment: positive

--- PRECEDING TEXT ---
Aber es liegt doch ei»nicht geringer Trost in der Wahrnehmung, daß, wenn cszur Entscheidung kommt, der Wähler das letzte Wort zusprechen hat, und daß auch jene Parteien, welche ihre Existenzans die Gunst deS Fürsten Bismarck stützen, der Instanznicht entbehre» können, die an der Urne ihren Willen verkündet.

--- RETRIEVED TEXT ---
Topic: 7
Original label: 7_wahlen_wähler_abstimmung_wahl
Category: CTX
Sentiment: positive

Diesen Trost gewährt die diesmalige Budget=Debatte des Reichstages, und dadurch empfängt sie ihre Bedeutung. Es ist von allen Parteien wie vom Regierungstische hinausgesprochen worden zu dem Volke; gute Vorsätze und verschämte Bekenntnisse, scharfe Anschuldigungen und beschwichtigende Friedensbethenerungen, kühne Beweisführungen und mühsame Widerlegungen — die bunte Fülle wird sich in dem Intellecte des Wählers von selbst sichten, und erst

In [19]:
results = search_newspapers(
    query="refugees",
    year=1905,
    sentiment ="negative",
    category = "MIG",
    k=20,
)

## Query 3

In [20]:
results = search_newspapers(
    query="education and minorities",
    sentiment= "negative",
    year=1884,
    k=10,
)

In [21]:
for number, result in enumerate(results, start=1):
    metadata = result.metadata

    print("=" * 100)
    print(f"Result {number}")
    print("ID:", metadata.get("chunk_id"))
    print("Date:", metadata.get("date"))
    print("Year:", metadata.get("year"))
    print("Newspaper:", metadata.get("newspaper_title"))
    print("Topic:", metadata.get("topic"))
    print("Category:", metadata.get("category"))
    print("Sentiment:", metadata.get("sentiment"))

    print("\n--- PRECEDING TEXT ---")
    print(metadata.get("preceding_document") or "[No preceding text]")

    print("\n--- RETRIEVED TEXT ---")
    print(result.page_content or "[No retrieved text]")

    print("\n--- FOLLOWING TEXT ---")
    print(metadata.get("following_document") or "[No following text]")

    print()

Result 1
ID: 177820.0
Date: 1884-08-01
Year: 1884
Newspaper: nfp
Topic: 3
Category: MIN
Sentiment: negative

--- PRECEDING TEXT ---
DaS ofsicielle Organ fährt dann, nachdem esden Wünschen der Italiener Tirols Ausdruck gegeben, in folgender Weise fort: „Ohne Bedenken müssen wir behauplen, daß alleSloveneu, auch die radicalsten, Gott auf den Knien danken wurden, wenn man der slovenischen Sache einmal so viele Concessionen vergönnen möchte, wie .solche so reichlich die Trientincrfett Langem in Frieden genießen.

--- RETRIEVED TEXT ---
Topic: 3
Original label: 3_czechen_polen_deutschen_czechischen
Category: MIN
Sentiment: negative

Die Czechen bekämpfen diese Forderungen der Wälschtiroler, zugleich aber auch, obwol indirect, das heiße Sehnen der Slovenen nach Vereinigung derselben zu einem staatsrechtlichen Ganzen

--- FOLLOWING TEXT ---
DaS vereinigteSlovsnien würde auf unsere nationale Entwicklung einen mächtigen und wohlthätigen Einfluß üben; mit der Förderung _ derslovenischen Sprache 

In [22]:
results = search_newspapers(
    query=(
        "Croats and Serbs; Croatian and Serbian populations, "
        "Croatien, Kroatien, Serbien, Serben, Kroaten, "
        "Serbo-Kroaten, Serbokroaten, Südslawen"
    ),
    year=1889,
    k=20,
)

In [23]:
for number, result in enumerate(results, start=1):
    metadata = result.metadata

    print("=" * 100)
    print(f"Result {number}")
    print("ID:", metadata.get("chunk_id"))
    print("Date:", metadata.get("date"))
    print("Year:", metadata.get("year"))
    print("Newspaper:", metadata.get("newspaper_title"))
    print("Topic:", metadata.get("topic"))
    print("Category:", metadata.get("category"))
    print("Sentiment:", metadata.get("sentiment"))

    print("\n--- PRECEDING TEXT ---")
    print(metadata.get("preceding_document") or "[No preceding text]")

    print("\n--- RETRIEVED TEXT ---")
    print(result.page_content or "[No retrieved text]")

    print("\n--- FOLLOWING TEXT ---")
    print(metadata.get("following_document") or "[No following text]")

    print()

Result 1
ID: 214937.0
Date: 1889-03-01
Year: 1889
Newspaper: nfp
Topic: 225
Category: CTX
Sentiment: positive

--- PRECEDING TEXT ---
Ichhoffe, eS wird nicht dahin kommen.

--- RETRIEVED TEXT ---
Topic: 225
Original label: 225_monarchie_dynastie_monarchisten_kaiserreich
Category: CTX
Sentiment: positive

Wir lieben unser Oesterreich treu und beharrlich, wir lieben Oesterreich, nicht das des Papstes, sondern das des Kaisers von Oesterreich, nicht die Raritätenkammer für czechtsches Staatsrecht und für hageilonische Ideen, sondern das Oesterreich, wie es auf gesetzlicher Grundlage gewachsen ist, sich aufgebaut hat und wie s weiter gedeihen nöge!

--- FOLLOWING TEXT ---
Weil wir Oesterreich lieben, darum sindwir entschieden und beharrlich gegen diese Regierung.

Result 2
ID: 117391.0
Date: 1889-08-01
Year: 1889
Newspaper: vtl
Topic: 0
Category: MIN
Sentiment: negative

--- PRECEDING TEXT ---
In jenem Vertrage ist nämlich festgesetzt, daß Derjenige, der das Gastrecht in einem der beiden St

In [24]:
def build_multilingual_query(
    english: str,                         # English search 
    german: str,                          # German search 
    historical_terms: list[str] | None = None,  # Optional historical variants
) -> str:

    # C=create the main English and German parts
    parts = [
        f"English concepts: {english.strip()}",
        f"German concepts: {german.strip()}",
    ]

    # add historical terms if they were provided.
    if historical_terms:
        parts.append(
            "Historical terms and spelling variants: "
            + ", ".join(historical_terms)
        )

    return "\n".join(parts)

In [25]:
query = build_multilingual_query(
    english=(
        "Croats and Serbs, ethnic relations, national identity, "
        "political conflict and minority rights"
    ),
    german=(
        "Kroaten und Serben, ethnische Beziehungen, nationale Identität, "
    ),
    historical_terms=[
        "Croaten",
        "Croatien",
        "Serbien",
        "Serben",
        "Südslawen",
    
    ],
)

results = search_newspapers(
    query=query,
    category="MIN",
    k=20,
)

In [26]:
for number, result in enumerate(results, start=1):
    metadata = result.metadata

    print("=" * 100)
    print(f"Result {number}")
    print("ID:", metadata.get("chunk_id"))
    print("Date:", metadata.get("date"))
    print("Year:", metadata.get("year"))
    print("Newspaper:", metadata.get("newspaper_title"))
    print("Topic:", metadata.get("topic"))
    print("Category:", metadata.get("category"))
    print("Sentiment:", metadata.get("sentiment"))

    print("\n--- PRECEDING TEXT ---")
    print(metadata.get("preceding_document") or "[No preceding text]")

    print("\n--- RETRIEVED TEXT ---")
    print(result.page_content or "[No retrieved text]")

    print("\n--- FOLLOWING TEXT ---")
    print(metadata.get("following_document") or "[No following text]")

    print()

Result 1
ID: 107166.0
Date: 1875-09-01
Year: 1875
Newspaper: nfp
Topic: 27
Category: MIN
Sentiment: negative

--- PRECEDING TEXT ---
Auch kann man wohl annehmen, daß sichOesterreich für den Fall, daß es im Interesse der Drei«Kaiser-Politik in die Action treten sollt?, sowol bezüglich derpolitischen Coitsequenzen, als der materiellen Opferder Schadloshaltung seitens seiner Verbündeten vergewissert habe."

--- RETRIEVED TEXT ---
Topic: 27
Original label: 27_bosnien_serben_serbien_montenegro
Category: MIN
Sentiment: negative

— Die Welt würde mit sehr wenig Weisheit regiert werden, wenn die drei Mächte den Fall eines serbischen Excesses nicht vorgesehen hätten, aber die Richtigkeit der obigen Angaben scheint uns denndoch im höchsten Grade zweifelhaft, und die Schlesische Zeitung thut Recht, diese Mittheilung mit einem skeptischen Fragezeichen zu breiten. Wien, .

--- FOLLOWING TEXT ---
(Preßstimmen über den Aufstand.)

Result 2
ID: 174556.0
Date: 1884-02-01
Year: 1884
Newspaper: nfp
Topic